In [2]:
import pandas as pd

pd.options.display.float_format = '{:.2f}'.format

# Carregar os arquivos de 2014
dre = pd.read_csv("../DRE_tratado/dfp_dre_2014_filtrado.csv",              sep=";", encoding="utf-8")
bpa = pd.read_csv("../Balanco_ativo_tratado/dfp_bp_2014_filtrado.csv",     sep=";", encoding="utf-8")
bpp = pd.read_csv("../Balanco_passivo_tratado/dfp_bpp_2014_filtrado.csv",  sep=";", encoding="utf-8")

# Filtrar setor energia
dre_en = dre[dre['setor'] == 'Energia']
bpa_en = bpa[bpa['setor'] == 'Energia']
bpp_en = bpp[bpp['setor'] == 'Energia']

print("Empresas na DRE:", dre_en['empresa'].unique())
print("Empresas no BPA:", bpa_en['empresa'].unique())
print("Empresas no BPP:", bpp_en['empresa'].unique())

Empresas na DRE: <StringArray>
[  'CENTRAIS ELET BRAS S.A. - ELETROBRAS',
                          'ENERGISA S.A.',
                        'NEOENERGIA S.A.',
                      'CPFL ENERGIA S.A.',
              'ENGIE BRASIL ENERGIA S.A.',
                'EQUATORIAL ENERGIA S.A.',
                'ALUPAR INVESTIMENTO S/A',
 'CIA ENERGETICA DE MINAS GERAIS - CEMIG',
      'CIA PARANAENSE DE ENERGIA - COPEL']
Length: 9, dtype: str
Empresas no BPA: <StringArray>
[  'CENTRAIS ELET BRAS S.A. - ELETROBRAS',
                          'ENERGISA S.A.',
                        'NEOENERGIA S.A.',
                      'CPFL ENERGIA S.A.',
              'ENGIE BRASIL ENERGIA S.A.',
                'EQUATORIAL ENERGIA S.A.',
                'ALUPAR INVESTIMENTO S/A',
 'CIA ENERGETICA DE MINAS GERAIS - CEMIG',
      'CIA PARANAENSE DE ENERGIA - COPEL']
Length: 9, dtype: str
Empresas no BPP: <StringArray>
[  'CENTRAIS ELET BRAS S.A. - ELETROBRAS',
                          'ENERGISA S.A.',
   

In [3]:
# Contas que precisamos para os 6 KPIs de Energia
contas_necessarias = [
    '3.01',     # Receita Líquida
    '3.02',     # Custos
    '3.03',     # Resultado Bruto
    '3.04.02',  # Despesas Gerais e Administrativas
    '3.06',     # Resultado Financeiro
    '3.06.02',  # Despesas Financeiras
    '3.11',     # Lucro/Prejuízo Consolidado
    '3.11.01',  # Atribuído ao Controlador
]

# Ver se todas as empresas têm essas contas
print("=== COBERTURA DAS CONTAS ===")
for conta in contas_necessarias:
    empresas = dre_en[dre_en['codigo_conta'] == conta]['empresa'].tolist()
    faltando = set(dre_en['empresa'].unique()) - set(empresas)
    if faltando:
        print(f"❌ {conta}: faltando {faltando}")
    else:
        print(f"✅ {conta}: todas as 9 empresas")

# Ver BPP — Patrimônio Líquido e Dívida
print("\n=== BPP — PATRIMÔNIO LÍQUIDO ===")
for conta in ['2.03', '2.08', '2.07']:
    empresas = bpp_en[bpp_en['codigo_conta'] == conta]['empresa'].tolist()
    if empresas:
        print(f"  {conta}: {empresas}")

=== COBERTURA DAS CONTAS ===
✅ 3.01: todas as 9 empresas
✅ 3.02: todas as 9 empresas
✅ 3.03: todas as 9 empresas
✅ 3.04.02: todas as 9 empresas
✅ 3.06: todas as 9 empresas
✅ 3.06.02: todas as 9 empresas
✅ 3.11: todas as 9 empresas
✅ 3.11.01: todas as 9 empresas

=== BPP — PATRIMÔNIO LÍQUIDO ===
  2.03: ['CENTRAIS ELET BRAS S.A. - ELETROBRAS', 'ENERGISA S.A.', 'NEOENERGIA S.A.', 'CPFL ENERGIA S.A.', 'ENGIE BRASIL ENERGIA S.A.', 'EQUATORIAL ENERGIA S.A.', 'ALUPAR INVESTIMENTO S/A', 'CIA ENERGETICA DE MINAS GERAIS - CEMIG', 'CIA PARANAENSE DE ENERGIA - COPEL']


In [4]:
# Para Dívida Líquida precisamos de:
# Dívida Bruta (empréstimos e financiamentos) — no BPP
# Caixa e Equivalentes — no BPA

print("=== BPP — DÍVIDA (empréstimos e financiamentos) ===")
termos = ['empréstimo', 'financiamento', 'debênture', 'dívida']
for termo in termos:
    contas = bpp_en[bpp_en['descricao_conta'].str.contains(termo, case=False, na=False)][['empresa', 'codigo_conta', 'descricao_conta']].drop_duplicates(subset=['codigo_conta', 'descricao_conta'])
    if len(contas) > 0:
        print(f"\n  '{termo}':")
        print(contas.to_string(index=False))

print("\n=== BPA — CAIXA ===")
termos_caixa = ['caixa', 'equivalente', 'aplicação financeira']
for termo in termos_caixa:
    contas = bpa_en[bpa_en['descricao_conta'].str.contains(termo, case=False, na=False)][['empresa', 'codigo_conta', 'descricao_conta']].drop_duplicates(subset=['codigo_conta', 'descricao_conta'])
    if len(contas) > 0:
        print(f"\n  '{termo}':")
        print(contas.to_string(index=False))

=== BPP — DÍVIDA (empréstimos e financiamentos) ===

  'empréstimo':
                             empresa  codigo_conta              descricao_conta
CENTRAIS ELET BRAS S.A. - ELETROBRAS       2.01.04 Empréstimos e Financiamentos
CENTRAIS ELET BRAS S.A. - ELETROBRAS    2.01.04.01 Empréstimos e Financiamentos
CENTRAIS ELET BRAS S.A. - ELETROBRAS 2.01.05.02.04       Empréstimo compulsório
CENTRAIS ELET BRAS S.A. - ELETROBRAS       2.02.01 Empréstimos e Financiamentos
CENTRAIS ELET BRAS S.A. - ELETROBRAS    2.02.01.01 Empréstimos e Financiamentos
CENTRAIS ELET BRAS S.A. - ELETROBRAS 2.02.02.02.03       Empréstimo compulsório

  'financiamento':
                             empresa  codigo_conta                                                 descricao_conta
CENTRAIS ELET BRAS S.A. - ELETROBRAS       2.01.04                                    Empréstimos e Financiamentos
CENTRAIS ELET BRAS S.A. - ELETROBRAS    2.01.04.01                                    Empréstimos e Financiamentos
CENTRA

In [5]:
# Buscar subcontas de empréstimos/financiamentos no circulante e não circulante
empresas_energia = dre_en['empresa'].unique()

for empresa in empresas_energia:
    print(f"\n=== {empresa} ===")
    contas = bpp_en[
        (bpp_en['empresa'] == empresa) &
        (bpp_en['codigo_conta'].str.match(r'^2\.0[12]\.\d'))
    ][['codigo_conta', 'descricao_conta', 'valor']]
    print(contas.sort_values('codigo_conta').to_string(index=False))


=== CENTRAIS ELET BRAS S.A. - ELETROBRAS ===
 codigo_conta                                              descricao_conta       valor
      2.01.01                            Obrigações Sociais e Trabalhistas        0.00
   2.01.01.01                                           Obrigações Sociais        0.00
   2.01.01.02                                      Obrigações Trabalhistas        0.00
      2.01.02                                                 Fornecedores  7489134.00
   2.01.02.01                                       Fornecedores Nacionais  7470482.00
   2.01.02.02                                    Fornecedores Estrangeiros    18652.00
      2.01.03                                           Obrigações Fiscais  1186306.00
   2.01.03.01                                  Obrigações Fiscais Federais  1186306.00
2.01.03.01.01               Imposto de Renda e Contribuição Social a Pagar  1168168.00
2.01.03.01.02                       Imposto de renda e contribuição social    18138.

In [6]:
# ============================================================
# EXTRAÇÃO DAS CONTAS — ENERGIA 2014
# ============================================================

# DRE
ll           = dre_en[dre_en['codigo_conta'] == '3.11.01'][['empresa', 'valor']].rename(columns={'valor': 'lucro_liquido'})
receita      = dre_en[dre_en['codigo_conta'] == '3.01'][['empresa', 'valor']].rename(columns={'valor': 'receita'})
custo        = dre_en[dre_en['codigo_conta'] == '3.02'][['empresa', 'valor']].rename(columns={'valor': 'custo'})
res_bruto    = dre_en[dre_en['codigo_conta'] == '3.03'][['empresa', 'valor']].rename(columns={'valor': 'resultado_bruto'})
desp_fin     = dre_en[dre_en['codigo_conta'] == '3.06.02'][['empresa', 'valor']].rename(columns={'valor': 'desp_financeiras'})
res_fin      = dre_en[dre_en['codigo_conta'] == '3.06'][['empresa', 'valor']].rename(columns={'valor': 'resultado_financeiro'})

# BPP
pl           = bpp_en[bpp_en['codigo_conta'] == '2.03'][['empresa', 'valor']].rename(columns={'valor': 'patrimonio_liquido'})
div_cp       = bpp_en[bpp_en['codigo_conta'] == '2.01.04'][['empresa', 'valor']].rename(columns={'valor': 'div_circulante'})
div_lp       = bpp_en[bpp_en['codigo_conta'] == '2.02.01'][['empresa', 'valor']].rename(columns={'valor': 'div_nao_circulante'})

# BPA
caixa        = bpa_en[bpa_en['codigo_conta'] == '1.01.01'][['empresa', 'valor']].rename(columns={'valor': 'caixa'})

# Verificar cobertura
print("=== COBERTURA ===")
for nome, df in [('lucro_liquido', ll), ('receita', receita), ('resultado_bruto', res_bruto),
                 ('desp_financeiras', desp_fin), ('pl', pl), ('div_cp', div_cp),
                 ('div_lp', div_lp), ('caixa', caixa)]:
    faltando = set(dre_en['empresa'].unique()) - set(df['empresa'].unique())
    if faltando:
        print(f"❌ {nome}: faltando {faltando}")
    else:
        print(f"✅ {nome}: todas as 9 empresas")

=== COBERTURA ===
✅ lucro_liquido: todas as 9 empresas
✅ receita: todas as 9 empresas
✅ resultado_bruto: todas as 9 empresas
✅ desp_financeiras: todas as 9 empresas
✅ pl: todas as 9 empresas
✅ div_cp: todas as 9 empresas
✅ div_lp: todas as 9 empresas
✅ caixa: todas as 9 empresas


In [7]:
# ============================================================
# CÁLCULO DOS KPIs — ENERGIA 2014
# ============================================================

# Montar dataframe base
kpis = ll.merge(receita,   on='empresa')
kpis = kpis.merge(custo,        on='empresa')
kpis = kpis.merge(res_bruto,    on='empresa')
kpis = kpis.merge(desp_fin,     on='empresa')
kpis = kpis.merge(res_fin,      on='empresa')
kpis = kpis.merge(pl,           on='empresa')
kpis = kpis.merge(div_cp,       on='empresa')
kpis = kpis.merge(div_lp,       on='empresa')
kpis = kpis.merge(caixa,        on='empresa')

# Calcular KPIs
kpis['roe']              = kpis['lucro_liquido'] / kpis['patrimonio_liquido']
kpis['margem_liquida']   = kpis['lucro_liquido'] / kpis['receita']
kpis['margem_bruta']     = kpis['resultado_bruto'] / kpis['receita']

# EBITDA aproximado = Resultado Bruto - Despesas Operacionais + Depreciação
# Como não temos depreciação separada ainda, usamos EBIT = Resultado antes do financeiro
# EBIT = Lucro Líquido - Resultado Financeiro + IR (aproximação)
# Simplificado: EBITDA ≈ Resultado Bruto + Despesas Gerais
desp_ga = dre_en[dre_en['codigo_conta'] == '3.04.02'][['empresa', 'valor']].rename(columns={'valor': 'desp_ga'})
kpis = kpis.merge(desp_ga, on='empresa')

# EBIT = Resultado antes do financeiro e tributos (conta 3.05)
ebit = dre_en[dre_en['codigo_conta'] == '3.05'][['empresa', 'valor']].rename(columns={'valor': 'ebit'})
kpis = kpis.merge(ebit, on='empresa')

# Dívida Líquida e Alavancagem
kpis['divida_bruta']     = kpis['div_circulante'] + kpis['div_nao_circulante']
kpis['divida_liquida']   = kpis['divida_bruta'] - kpis['caixa']

# EBITDA ≈ EBIT + Depreciação — vamos buscar depreciação
depr = dre_en[dre_en['codigo_conta'] == '3.04.02.08'][['empresa', 'valor']].rename(columns={'valor': 'depreciacao'})
kpis = kpis.merge(depr, on='empresa', how='left')
kpis['depreciacao'] = kpis['depreciacao'].fillna(0)
kpis['ebitda']           = kpis['ebit'] + kpis['depreciacao'].abs()

kpis['margem_ebitda']    = kpis['ebitda'] / kpis['receita']
kpis['alavancagem']      = kpis['divida_liquida'] / kpis['ebitda']
kpis['cobertura_juros']  = kpis['ebit'] / kpis['desp_financeiras'].abs()

# Adicionar setor e tipo
import sys
import os

# Pega o caminho da pasta atual do notebook e sobe um nível ('..')
caminho_raiz = os.path.abspath(os.path.join(os.getcwd(), '..'))

# Adiciona a pasta raiz ao sys.path, se já não estiver lá
if caminho_raiz not in sys.path:
    sys.path.append(caminho_raiz)

# Agora a importação vai funcionar normalmente
from empresas_selecionadas import empresas_selecionadas
kpis['setor'] = kpis['empresa'].map(lambda x: empresas_selecionadas[x][0]) # type: ignore
kpis['tipo']  = kpis['empresa'].map(lambda x: empresas_selecionadas[x][1]) # type: ignore

# Exibir resultado
cols = ['empresa', 'tipo', 'roe', 'margem_liquida', 'margem_bruta', 
        'margem_ebitda', 'alavancagem', 'cobertura_juros']
print(kpis[cols].sort_values('roe', ascending=False).to_string(index=False))

                               empresa    tipo   roe  margem_liquida  margem_bruta  margem_ebitda  alavancagem  cobertura_juros
CIA ENERGETICA DE MINAS GERAIS - CEMIG Estatal  0.28            0.16          0.34           0.29         2.26             3.29
             ENGIE BRASIL ENERGIA S.A. Privado  0.24            0.21          0.39           0.36         1.06             4.17
               EQUATORIAL ENERGIA S.A. Privado  0.19            0.09          0.25           0.14         4.44             0.83
                     CPFL ENERGIA S.A. Privado  0.10            0.05          0.23           0.15         5.69             1.31
               ALUPAR INVESTIMENTO S/A Privado  0.09            0.25          0.77           0.71         3.46             3.34
                         ENERGISA S.A. Privado  0.09            0.03          0.23           0.12         6.93             0.85
     CIA PARANAENSE DE ENERGIA - COPEL Estatal  0.09            0.09          0.20           0.12       

In [8]:
# Ver detalhes da Eletrobras
eletro = kpis[kpis['empresa'] == 'CENTRAIS ELET BRAS S.A. - ELETROBRAS']

print("=== ELETROBRAS — VALORES BRUTOS ===")
print(f"Lucro Líquido:    {eletro['lucro_liquido'].values[0]:,.0f}")
print(f"Receita:          {eletro['receita'].values[0]:,.0f}")
print(f"EBIT:             {eletro['ebit'].values[0]:,.0f}")
print(f"Depreciação:      {eletro['depreciacao'].values[0]:,.0f}")
print(f"EBITDA:           {eletro['ebitda'].values[0]:,.0f}")
print(f"Dívida Líquida:   {eletro['divida_liquida'].values[0]:,.0f}")
print(f"Desp Financeiras: {eletro['desp_financeiras'].values[0]:,.0f}")

# Ver conta 3.04.02.08 da Eletrobras
print("\n=== DEPRECIAÇÃO ELETROBRAS ===")
print(dre_en[
    (dre_en['empresa'] == 'CENTRAIS ELET BRAS S.A. - ELETROBRAS') &
    (dre_en['codigo_conta'].str.startswith('3.04.02'))
][['codigo_conta', 'descricao_conta', 'valor']].to_string(index=False))

=== ELETROBRAS — VALORES BRUTOS ===
Lucro Líquido:    -3,031,055
Receita:          30,244,854
EBIT:             -1,956,609
Depreciação:      -1,777,296
EBITDA:           -179,313
Dívida Líquida:   38,475,106
Desp Financeiras: -4,511,129

=== DEPRECIAÇÃO ELETROBRAS ===
codigo_conta                                    descricao_conta        valor
     3.04.02                  Despesas Gerais e Administrativas -14657264.00
  3.04.02.01                         Pessoal, Material, Serviço  -8485373.00
  3.04.02.02                       Resultado a compensar Itaipu         0.00
  3.04.02.06                        Remuneração e Ressarcimento   -386824.00
  3.04.02.08 Depreciação e amortizaçao-Imobilizado e Intangível  -1777296.00
  3.04.02.09                             Provisões operacionais  -1861707.00
  3.04.02.10                            Doações e contribuições   -251415.00
  3.04.02.11          Plano de readequação do quadro de pessoal   -219299.00
  3.04.02.12                          

In [9]:
# Recalcular alavancagem com tratamento para EBITDA negativo
kpis['alavancagem'] = kpis.apply(
    lambda r: r['divida_liquida'] / r['ebitda'] if r['ebitda'] > 0 else None,
    axis=1
)

# Recalcular cobertura de juros com tratamento para EBIT negativo  
kpis['cobertura_juros'] = kpis.apply(
    lambda r: r['ebit'] / abs(r['desp_financeiras']) if r['desp_financeiras'] != 0 else None,
    axis=1
)

# Exibir resultado corrigido
cols = ['empresa', 'tipo', 'roe', 'margem_liquida', 'margem_bruta',
        'margem_ebitda', 'alavancagem', 'cobertura_juros']
print(kpis[cols].sort_values('roe', ascending=False).to_string(index=False))

                               empresa    tipo   roe  margem_liquida  margem_bruta  margem_ebitda  alavancagem  cobertura_juros
CIA ENERGETICA DE MINAS GERAIS - CEMIG Estatal  0.28            0.16          0.34           0.29         2.26             3.29
             ENGIE BRASIL ENERGIA S.A. Privado  0.24            0.21          0.39           0.36         1.06             4.17
               EQUATORIAL ENERGIA S.A. Privado  0.19            0.09          0.25           0.14         4.44             0.83
                     CPFL ENERGIA S.A. Privado  0.10            0.05          0.23           0.15         5.69             1.31
               ALUPAR INVESTIMENTO S/A Privado  0.09            0.25          0.77           0.71         3.46             3.34
                         ENERGISA S.A. Privado  0.09            0.03          0.23           0.12         6.93             0.85
     CIA PARANAENSE DE ENERGIA - COPEL Estatal  0.09            0.09          0.20           0.12       

In [10]:
eventos_exogenos = {
    ('CENTRAIS ELET BRAS S.A. - ELETROBRAS', 2014): 'Lei 12.783/2013 - Renovação tarifária',
    ('CENTRAIS ELET BRAS S.A. - ELETROBRAS', 2015): 'Lei 12.783/2013 - Renovação tarifária',
}

kpis['evento_exogeno']   = 0
kpis['descricao_evento'] = ''

for (empresa, ano_ev), descricao in eventos_exogenos.items():
    mask = (kpis['empresa'] == empresa)
    kpis.loc[mask, 'evento_exogeno']   = 1
    kpis.loc[mask, 'descricao_evento'] = descricao

print(kpis[['empresa', 'tipo', 'evento_exogeno', 'descricao_evento']].to_string(index=False))

                               empresa    tipo  evento_exogeno                      descricao_evento
  CENTRAIS ELET BRAS S.A. - ELETROBRAS Estatal               1 Lei 12.783/2013 - Renovação tarifária
                         ENERGISA S.A. Privado               0                                      
                       NEOENERGIA S.A. Privado               0                                      
                     CPFL ENERGIA S.A. Privado               0                                      
             ENGIE BRASIL ENERGIA S.A. Privado               0                                      
               EQUATORIAL ENERGIA S.A. Privado               0                                      
               ALUPAR INVESTIMENTO S/A Privado               0                                      
CIA ENERGETICA DE MINAS GERAIS - CEMIG Estatal               0                                      
     CIA PARANAENSE DE ENERGIA - COPEL Estatal               0                             

In [11]:
from scipy import stats

kpis_config_energia = {
    'roe':             True,
    'margem_liquida':  True,
    'margem_bruta':    True,
    'margem_ebitda':   True,
    'alavancagem':     False,
    'cobertura_juros': True,
}

privados = kpis[kpis['tipo'] == 'Privado']
estatais = kpis[kpis['tipo'] == 'Estatal']

linhas = []
for estatal in estatais.itertuples():
    for kpi, maior_melhor in kpis_config_energia.items():
        # Usar só privados que têm valor válido para esse KPI
        valores_privados = privados[kpi].dropna()
        media  = valores_privados.mean()
        desvio = valores_privados.std()
        valor  = getattr(estatal, kpi)

        if pd.isna(valor) or desvio == 0:
            zscore = None
            sinal  = '⚠️ Dado indisponível'
        else:
            zscore = (valor - media) / desvio
            zscore_int = zscore if maior_melhor else -zscore

            if zscore_int > 0.5:
                sinal = 'Acima da média'
            elif zscore_int < -1:
                sinal = 'Alerta'
            elif zscore_int < -0.5:
                sinal = 'Abaixo da média'
            else:
                sinal = 'Na média'

        linhas.append({
            'empresa':          estatal.empresa,
            'tipo':             estatal.tipo,
            'kpi':              kpi,
            'valor':            round(valor, 2) if not pd.isna(valor) else None,
            'media_privados':   round(media, 2),
            'desvio_padrao':    round(desvio, 2),
            'zscore':           round(zscore, 2) if zscore is not None else None,
            'sinal':            sinal,
            'evento_exogeno':   estatal.evento_exogeno,
            'descricao_evento': estatal.descricao_evento,
        })

zscores = pd.DataFrame(linhas)
print(zscores.to_string(index=False))

                               empresa    tipo             kpi  valor  media_privados  desvio_padrao  zscore                sinal  evento_exogeno                      descricao_evento
  CENTRAIS ELET BRAS S.A. - ELETROBRAS Estatal             roe  -0.05            0.13           0.07   -2.67               Alerta               1 Lei 12.783/2013 - Renovação tarifária
  CENTRAIS ELET BRAS S.A. - ELETROBRAS Estatal  margem_liquida  -0.10            0.12           0.09   -2.38               Alerta               1 Lei 12.783/2013 - Renovação tarifária
  CENTRAIS ELET BRAS S.A. - ELETROBRAS Estatal    margem_bruta   0.46            0.35           0.21    0.51       Acima da média               1 Lei 12.783/2013 - Renovação tarifária
  CENTRAIS ELET BRAS S.A. - ELETROBRAS Estatal   margem_ebitda  -0.01            0.27           0.24   -1.16               Alerta               1 Lei 12.783/2013 - Renovação tarifária
  CENTRAIS ELET BRAS S.A. - ELETROBRAS Estatal     alavancagem    NaN           

In [12]:
import pandas as pd

kpis_en = pd.read_csv("../KPIs/kpis_energia.csv", sep=";", encoding="utf-8")

pivot = kpis_en[kpis_en['tipo'] == 'Privado'].groupby(['empresa', 'ano']).size().unstack(fill_value=0)
print(pivot.to_string())

ano                                   2014  2015  2016  2017  2018  2019  2020  2021  2022  2023
empresa                                                                                         
ALUPAR INVESTIMENTO S/A                  1     1     1     1     1     1     1     1     1     1
CENTRAIS ELET BRAS S.A. - ELETROBRAS     0     0     0     0     0     0     0     0     1     1
CIA PARANAENSE DE ENERGIA - COPEL        0     0     0     0     0     0     0     0     0     1
CPFL ENERGIA S.A.                        1     1     1     1     1     1     1     1     1     1
ENERGISA S.A.                            1     1     1     1     1     1     1     1     1     1
ENGIE BRASIL ENERGIA S.A.                1     1     1     1     1     1     1     1     1     1
EQUATORIAL ENERGIA S.A.                  1     1     1     1     1     1     1     1     1     1
NEOENERGIA S.A.                          1     1     1     1     1     1     1     1     1     1


In [13]:
dre_2020 = pd.read_csv("../DRE_tratado/dfp_dre_2020_filtrado.csv", sep=";", encoding="utf-8")
bpa_2020 = pd.read_csv("../Balanco_ativo_tratado/dfp_bp_2020_filtrado.csv", sep=";", encoding="utf-8")
bpp_2020 = pd.read_csv("../Balanco_passivo_tratado/dfp_bpp_2020_filtrado.csv", sep=";", encoding="utf-8")

dre_eq = dre_2020[dre_2020['empresa'] == 'EQUATORIAL ENERGIA S.A.']
bpp_eq = bpp_2020[bpp_2020['empresa'] == 'EQUATORIAL ENERGIA S.A.']
bpa_eq = bpa_2020[bpa_2020['empresa'] == 'EQUATORIAL ENERGIA S.A.']

print("=== EQUATORIAL DRE 2020 — contas relevantes ===")
contas = ['3.01', '3.03', '3.05', '3.06.02', '3.11.01']
print(dre_eq[dre_eq['codigo_conta'].isin(contas)][['codigo_conta', 'descricao_conta', 'valor']].to_string())

print("\n=== EQUATORIAL BPP 2020 — PL e Dívida ===")
contas_bpp = ['2.03', '2.01.04', '2.02.01']
print(bpp_eq[bpp_eq['codigo_conta'].isin(contas_bpp)][['codigo_conta', 'descricao_conta', 'valor']].to_string())

print("\n=== EQUATORIAL BPA 2020 — Caixa ===")
print(bpa_eq[bpa_eq['codigo_conta'] == '1.01.01'][['codigo_conta', 'descricao_conta', 'valor']].to_string())

=== EQUATORIAL DRE 2020 — contas relevantes ===
    codigo_conta                                         descricao_conta       valor
251         3.01                  Receita de Venda de Bens e/ou Serviços 17890069.00
253         3.03                                         Resultado Bruto  5986755.00
265         3.05  Resultado Antes do Resultado Financeiro e dos Tributos  4782427.00
268      3.06.02                                    Despesas Financeiras -1550847.00
278      3.11.01              Atribuído a Sócios da Empresa Controladora  2975089.00

=== EQUATORIAL BPP 2020 — PL e Dívida ===
    codigo_conta                 descricao_conta       valor
696      2.01.04    Empréstimos e Financiamentos  3112366.00
738      2.02.01    Empréstimos e Financiamentos 14675612.00
784         2.03  Patrimônio Líquido Consolidado 12278487.00

=== EQUATORIAL BPA 2020 — Caixa ===
    codigo_conta                descricao_conta      valor
465      1.01.01  Caixa e Equivalentes de Caixa 2219546.00


In [14]:
import pandas as pd

# Ver todos os nomes no arquivo bruto de 2020
dre_bruto = pd.read_csv("../DRE_bruto/dfp_dre_2020.csv", sep=";", encoding="latin1")

# Buscar qualquer coisa com "equatorial"
matches = dre_bruto[dre_bruto['DENOM_CIA'].str.contains('EQUATORIAL', case=False, na=False)]['DENOM_CIA'].unique()
print("Nomes com EQUATORIAL em 2020:")
print(matches)

Nomes com EQUATORIAL em 2020:
<StringArray>
['EQUATORIAL S.A.']
Length: 1, dtype: str


In [15]:
import pandas as pd

for ano in range(2014, 2024):
    try:
        dre = pd.read_csv(f"../DRE_bruto/dfp_dre_{ano}.csv", sep=";", encoding="latin1")
        matches = dre[dre['DENOM_CIA'].str.contains('EQUATORIAL', case=False, na=False)]['DENOM_CIA'].unique()
        print(f"{ano}: {list(matches)}")
    except:
        print(f"{ano}: erro ao ler")

2014: ['EQUATORIAL ENERGIA S.A.']
2015: ['EQUATORIAL ENERGIA S.A.']
2016: ['EQUATORIAL ENERGIA S.A.']
2017: ['EQUATORIAL ENERGIA S.A.']
2018: ['EQUATORIAL ENERGIA S.A.']
2019: ['EQUATORIAL ENERGIA S.A.']
2020: ['EQUATORIAL S.A.']
2021: ['EQUATORIAL S.A.']
2022: ['EQUATORIAL S.A.']
2023: ['EQUATORIAL S.A.']


In [16]:
import pandas as pd

for ano in [2022, 2023]:
    dre = pd.read_csv(f"../DRE_bruto/dfp_dre_{ano}.csv", sep=";", encoding="latin1")
    matches = dre[dre['DENOM_CIA'].str.contains('ELET', case=False, na=False)]['DENOM_CIA'].unique()
    print(f"{ano}: {list(matches)}")

2022: ['ELETRORIVER S.A.', 'ELETROMIDIA S.A.', 'INTELBRAS S.A. IND. DE TELECOMUNICAÃ\x87Ã\x83O ELETRÃ\x94NICA BRASILEIRA', 'CENTRAIS ELET DE SANTA CATARINA S.A.']
2023: ['ELETRORIVER S.A.', 'ELETROMIDIA S.A.', 'INTELBRAS S.A. IND. DE TELECOMUNICAÃ\x87Ã\x83O ELETRÃ\x94NICA BRASILEIRA', 'CENTRAIS ELET DE SANTA CATARINA S.A.']


In [17]:
for ano in [2022, 2023]:
    dre = pd.read_csv(f"../DRE_bruto/dfp_dre_{ano}.csv", sep=";", encoding="latin1")
    matches = dre[dre['DENOM_CIA'].str.contains('ELETROBRAS|CENTRAIS ELET BRAS', case=False, na=False)]['DENOM_CIA'].unique()
    print(f"{ano}: {list(matches)}")
    
    # Buscar pelo código CVM da Eletrobras (2437)
    matches_cvm = dre[dre['CD_CVM'] == 2437]['DENOM_CIA'].unique()
    print(f"{ano} (por código CVM 2437): {list(matches_cvm)}")

2022: []
2022 (por código CVM 2437): ['AXIA ENERGIA S.A.']
2023: []
2023 (por código CVM 2437): ['AXIA ENERGIA S.A.']
